In [40]:
import os
from dotenv import load_dotenv
load_dotenv()

groq_api_key = os.getenv("GROQ_API_KEY")

In [24]:
from langchain_groq import ChatGroq

# List of models to try (in order of preference)
# Check https://console.groq.com/docs/models for latest available models
models_to_try = [
    "llama-3.3-70b-versatile",
    "llama-3.2-90b-vision-preview", 
    "llama-3.2-70b-versatile",
    "llama-3.1-8b-instant",
    "mixtral-8x7b-32768",
]

model = None
for model_name in models_to_try:
    try:
        model = ChatGroq(model=model_name, groq_api_key=groq_api_key)
        print(f"✅ Using model: {model_name}")
        break
    except Exception as e:
        print(f"❌ {model_name} not available")
        continue

if model is None:
    print("⚠️ No models available! Check https://console.groq.com/docs/models for current models")
    print("And update the models_to_try list above with available models")
else:
    model

✅ Using model: llama-3.3-70b-versatile


In [33]:
from langchain_core.messages import HumanMessage,SystemMessage
messages = [
    SystemMessage(content="You are a helpful translator. Translate the following English text to French. Only provide the translated text, without any explanation or additional commentary."),
    HumanMessage(content="Hello How are you?")
]

result = model.invoke(messages)
print(result.content)

Bonjour, comment allez-vous ?


In [34]:
from langchain_core.output_parsers import StrOutputParser
parser = StrOutputParser()
parser.invoke(result)

'Bonjour, comment allez-vous ?'

In [35]:
### Using LCEL - chain the components
chain = model|parser
chain.invoke(messages)


'Bonjour, comment allez-vous ?'

In [36]:
### Prompt Templates
from langchain_core.prompts import ChatPromptTemplate

generic_template = "You are a professional translator. Translate the following text to {language}. Only provide the translation without explanation:"
prompt = ChatPromptTemplate.from_messages(
    [("system", generic_template), ("user", "{text}")]
)

In [37]:
result = prompt.invoke({"language":"French","text":"Hello"})

In [38]:
result.to_messages()

[SystemMessage(content='You are a professional translator. Translate the following text to French. Only provide the translation without explanation:', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='Hello', additional_kwargs={}, response_metadata={})]

In [39]:
chain=prompt|model|parser
chain.invoke({"language":"French","text":"Hello"})

'Bonjour'